In [1]:
import os

# Set the environment variable to a higher timeout (e.g., 2 seconds)
os.environ['PYDEVD_WARN_SLOW_RESOLVE_TIMEOUT'] = '2.0'

In [2]:
import mne
import numpy as np
import scipy.signal as signal
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import simps
from tqdm.notebook import tqdm
import os
from scipy.stats import skew, kurtosis, entropy
import scipy.stats as stats
from scipy.signal import find_peaks
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import spearmanr

In [4]:
# create folder to store result if not exist
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing"
if not os.path.exists(result_path):
    os.makedirs(result_path)

In [5]:
df_skill = pd.read_csv(f"C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data_skill.csv")
df_filtered = df_skill[["Participant", "Algorithm", "SkillScore", "EEG", "CrossEEG"]]
df_skill = df_skill[["Participant", "SkillScore"]]
df_skill = df_skill.drop_duplicates()
df_filtered

,Participant,Algorithm,SkillScore,EEG,CrossEEG
0,1,IsPrime,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1,1,SiebDesEratosthenes,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
2,1,IsAnagram,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
3,1,RemoveDoubleChar,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
4,1,BinToDecimal,0.331385,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
...,...,...,...,...,...
1067,71,DumpSorting,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1068,71,BinomialCoefficient,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1069,71,IsAnagram,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...
1070,71,ArrayAverage,0.435651,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...,C:/Users/Mahima Acharya/Documents/BCI/Data/eeg...


### Skill level assignment

- less than 0.33 quantile --> Novice  
- more than 0.66 quantile --> Expert  
- else --> Intermediate  

In [6]:
quantile = df_skill["SkillScore"].quantile([0.33, 0.66])
lower = quantile[0.33]
upper = quantile[0.66]

df_skill['SkillLevel'] = np.select([df_skill['SkillScore'] < lower, (df_skill['SkillScore'] >= lower) & (df_skill['SkillScore'] <= upper), df_skill['SkillScore'] > upper],
                                 ['Novice', 'Intermediate', 'Expert'],
                                 default='Intermediate')
df_skilled = df_skill.copy()
df_skilled

,Participant,SkillScore,SkillLevel
0,1,0.331385,Intermediate
32,2,0.379187,Expert
64,3,0.311264,Intermediate
80,4,0.424727,Expert
112,5,0.313031,Intermediate
144,6,0.315932,Intermediate
173,7,0.420873,Expert
205,10,0.350392,Expert
237,11,0.178206,Novice
261,12,0.309233,Intermediate


In [7]:
df_skill.groupby('SkillLevel')['Participant'].count()

SkillLevel
Expert          13
Intermediate    12
Novice          12
Name: Participant, dtype: int64

### Functions to calculate Brainwaves

In [8]:
def _to_decibel(spec):
    return 10 * np.log10(spec)

def get_spectrum(data, sampling_rate, method='welch', decibel=False, resolution='auto'):
    """
    Calculate amplitude or power spectrum

    data: Should be of shape (n_channels, n_samples)
    sampling_rate: Sampling rate... (float)
    method:
        * welch for power spectrum using Welch's method (recommended)
        * ft for simple Fourier transform (amplitude spectrum)
        * ps for power spectrum using a simple fourier transform
    decibel: Convert spectrum to decibel (bool)
    """

    axis = -1

    n_channels, n_samples = data.shape

    if resolution == 'auto':
        n_frequencies = n_samples
    elif isinstance(resolution, (int, float)):
        n_frequencies = np.round(sampling_rate / resolution).astype('int')
    else:
        raise ValueError('\'{}\''.format(resolution))

    # Spectrum
    if method in ['ft', 'ps']:
        # Using FFT
        # Get (complex) spectrum
        spec = np.fft.fft(data, n=n_frequencies, axis=axis)
        freq = np.fft.fftfreq(n_frequencies) * sampling_rate

        # Convert to real positive-sided spectrum
        spec = np.abs(spec)
        nyquist = 0.5 * sampling_rate
        is_positive = np.logical_or(np.logical_and(freq >= 0, freq <= nyquist), freq == -nyquist)
        n_pos = np.sum(is_positive)
        is_positive = np.repeat(is_positive[np.newaxis], n_channels, axis=0)
        spec = np.reshape(spec[is_positive], (n_channels, n_pos))
        freq = np.abs(freq[is_positive])
        is_double = np.logical_and(freq > 0, freq < nyquist)
        is_double = np.repeat(is_double[np.newaxis], n_channels, axis=0)
        spec[is_double] = 2 * spec[is_double]

        if method in ['ps']:
            # Get power spectral density
            spec = (1 / (sampling_rate * n_frequencies)) * spec ** 2

            # Convert to decibel if required
            if decibel:
                spec = _to_decibel(spec)
    elif method in ['welch', 'welch_db']:
        # Using Welch method
        freq, spec = signal.welch(data, sampling_rate, nperseg=n_frequencies, detrend='constant', axis=axis)

        # Convert to decibel if required
        if decibel:
            spec = _to_decibel(spec)
    else:
        raise RuntimeError('Unknown method \'{}\''.format(method))

    return spec, freq

In [9]:
#to extract the power within a specified frequency band from a power spectrum, 
#and  optionally normalizes the result if relative is set to True

def bandpower(spec, freq, freqband, relative=False):
    """
    Get band power within specified frequency band
    Alternatively: https://raphaelvallat.com/bandpower.html
    """

    spec = np.asarray(spec)
    freq = np.asarray(freq)
    freqband = np.asarray(freqband)

    if spec.ndim != 1:
        raise ValueError('Input \'spec\' bad: {}'.format(spec.shape))

    if freqband.ndim != 1 and freqband.shape[-1] != 2:
        raise ValueError('Input \'freqband\' bad: {}'.format(freqband.shape))

    # Frequency resolution
    step_freq = freq[1] - freq[0]

    # Find closest indices of band in frequency vector
    is_in_freqband = np.logical_and(freq >= np.min(freqband), freq <= np.max(freqband))

    # Integral approximation of the spectrum using Simpson's rule
    bp = simps(spec[is_in_freqband], dx=step_freq)

    if relative:
        bp = bp / simps(spec, dx=step_freq)

    return bp

In [14]:
def brainwaves(eeg_path, sampling_rate=500):
    # read in eeg file
    eeg_data = mne.io.read_raw_fif(eeg_path, preload=True, verbose='ERROR')

    # get raw channel data and do mean average referencing
    eeg_data_raw = eeg_data.get_data()
    eeg_data_ref = eeg_data_raw - np.mean(eeg_data_raw, axis=0)

    # extract channel names of eeg_data
    channel_names = list(eeg_data.to_data_frame().columns[1:])

    # create mock events for cutting eeg data (number, len, id)
    events = np.array([(0, 0, 1)])

    # create temporal eeg raw for cutting data into epochs
    tmp_raw = mne.io.RawArray(eeg_data_ref, eeg_data.info, verbose='ERROR')

    # Considered min. duration of the Participant's code comprehension as the time window
    time_window = 4

    # calculate EEG Signal duration
    eeg_duration = tmp_raw.n_times / tmp_raw.info['sfreq']

    # calculate midpoint of the signal
    midpoint = eeg_duration/2

    #Set time window starting and end point to choose the middle part of the EEG signal
    tmin = midpoint - (time_window/2)
    tmax = midpoint + (time_window/2)
    
    # create epochs which have a duration second window and operate on event id 1
    epochs = mne.Epochs(tmp_raw, events, event_id=1, tmin=tmin, tmax=tmax,baseline=None, preload=True, verbose='ERROR')
    #first_epoch = epochs[0]
    alpha = np.array([])
    beta = np.array([])
    gamma = np.array([])
    theta = np.array([])
    range_4_to_50 = np.array([])

    for epoch_data_raw in epochs:
    # perform power spectrum analysis on eeg data
        spectrum, frequency = get_spectrum(epoch_data_raw, sampling_rate, method='welch', decibel=False,
                                        resolution='auto')

        alpha_current = np.array([])
        beta_current = np.array([])
        gamma_current = np.array([])
        theta_current = np.array([])
        range_4_to_50_current = np.array([])

        for channel in channel_names:
            alpha_current = np.append(alpha_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [8.0, 13.0],
                                                relative=True))
            beta_current = np.append(beta_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [13.0, 30.0],
                                            relative=True))
            gamma_current = np.append(gamma_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [30.0, 50.0],
                                                relative=True))
            theta_current = np.append(theta_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [4.0, 8.0],
                                                relative=True))
            range_4_to_50_current = np.append(range_4_to_50_current,
                                    bandpower(spectrum[channel_names.index(channel), :], frequency, [4.0, 50.0],
                                                relative=True))

        alpha = np.append(alpha, alpha_current)
        beta = np.append(beta, beta_current)
        gamma = np.append(gamma, gamma_current)
        theta = np.append(theta, theta_current)
        range_4_to_50 = np.append(range_4_to_50, range_4_to_50_current)

    return alpha, beta, gamma, theta, range_4_to_50

### Brainwaves calculation

In [15]:
# read in montage and calculate the electrode positions
df_skill = pd.read_csv(f"C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data_skill.csv")

df_eeg_data = df_skill[["Participant", "Algorithm", "SkillScore", "EEG", "CrossEEG"]]

# sampling rate of the EEG data
sampling_rate = 500

# create Mental Workload column
df_brain_waves = pd.DataFrame(columns=["Participant", "SkillScore", "Algorithm", "Alpha", "Beta", "Gamma", "Theta", "Range4to50Hz","BaselineAlpha", "BaselineBeta", "BaselineGamma", "BaselineTheta", "BaselineRange4to50Hz"])

# iterate over each row anc calculate barin waves for the task
for idx in tqdm(range(len(df_filtered))):
    participant = df_filtered.iloc[idx]["Participant"]

    algorithm = df_filtered.iloc[idx]["Algorithm"]
    skill_score = df_filtered.iloc[idx]["SkillScore"]
    eeg_path = df_filtered.iloc[idx]["EEG"]
    cross_eeg_path = df_filtered.iloc[idx]["CrossEEG"]

    alpha, beta, gamma, theta, range4to50 = brainwaves(eeg_path)
    

    baseline_alpha, baseline_beta, baseline_gamma, baseline_theta, baseline_range4to50 = brainwaves(cross_eeg_path)
    alpha = alpha.reshape(64, 1)
    beta = beta.reshape(64, 1)
    gamma = gamma.reshape(64, 1)
    theta = theta.reshape(64, 1)
    range4to50 = range4to50.reshape(64,1)
    
    baseline_alpha = baseline_alpha.reshape(64, 1)
    baseline_beta = baseline_beta.reshape(64, 1)
    baseline_gamma = baseline_gamma.reshape(64, 1)
    baseline_theta = baseline_theta.reshape(64, 1)
    baseline_range4to50 = baseline_range4to50.reshape(64,1)


    df_brain_waves.loc[len(df_brain_waves)] = [participant, skill_score, algorithm, alpha, beta, gamma, theta, range4to50, baseline_alpha, baseline_beta, baseline_gamma, baseline_theta, baseline_range4to50]
 
df_brain_waves.to_csv(result_path+"/BrainWaves.csv")


  0%|          | 0/1072 [00:00<?, ?it/s]

In [16]:
df_brain_waves

,Participant,SkillScore,Algorithm,Alpha,Beta,Gamma,Theta,Range4to50Hz,BaselineAlpha,BaselineBeta,BaselineGamma,BaselineTheta,BaselineRange4to50Hz
0,1,0.331385,IsPrime,"[[0.1778556627649099], [0.1294300454494536], [...","[[0.12785309461468938], [0.17481799804998913],...","[[0.07321136743334146], [0.08930578645043821],...","[[0.22873350091444938], [0.1873401960575567], ...","[[0.6076536257273902], [0.5808940260074379], [...","[[0.12893736631902714], [0.13052248391744597],...","[[0.1949327305938576], [0.2207488607445728], [...","[[0.05891106149513782], [0.10684924675674091],...","[[0.10238700518909745], [0.14285755578310447],...","[[0.4934974808769842], [0.605755140731115], [0..."
1,1,0.331385,SiebDesEratosthenes,"[[0.14399128899708588], [0.14083532302932217],...","[[0.17098830181532818], [0.18959792215957177],...","[[0.09960026421162532], [0.06637512989532202],...","[[0.09127189594337842], [0.16889922549878147],...","[[0.5074623755120865], [0.5770144398756503], [...","[[0.2973188375069371], [0.3396716104122783], [...","[[0.1334035676382684], [0.11980772650574263], ...","[[0.0678748617080782], [0.0716394708637403], [...","[[0.1019550621165257], [0.10428815863177555], ...","[[0.6333511677018893], [0.6724246817175873], [..."
2,1,0.331385,IsAnagram,"[[0.1295553937970955], [0.09282823877611668], ...","[[0.16554861150430653], [0.15001045660538426],...","[[0.07485087416136596], [0.08269131237739936],...","[[0.19058703508450428], [0.2601196234617058], ...","[[0.5673785866341523], [0.6079983141209369], [...","[[0.39693865402301315], [0.33341281436728526],...","[[0.13301306960007464], [0.18685213113646118],...","[[0.047830021245293564], [0.07456727192237615]...","[[0.13586575073301865], [0.08082012108699872],...","[[0.7335055798110417], [0.6977508092089181], [..."
3,1,0.331385,RemoveDoubleChar,"[[0.17094104741567773], [0.15124354622812303],...","[[0.1268823216723157], [0.159300180041804], [0...","[[0.061578963694041575], [0.05898304983434938]...","[[0.24139148749165498], [0.2645480523689333], ...","[[0.6276414818034795], [0.6662657112007767], [...","[[0.3656780614728621], [0.2924782015564169], [...","[[0.12628383741052132], [0.16834485494551116],...","[[0.06306193457649979], [0.08659003328539422],...","[[0.08198915510632611], [0.06843162002288498],...","[[0.6663144321737743], [0.6223935196646638], [..."
4,1,0.331385,BinToDecimal,"[[0.17921206709946502], [0.1407427079432058], ...","[[0.16396297808006471], [0.20131184220965245],...","[[0.1155854466642323], [0.12052159022409228], ...","[[0.1731292208812361], [0.25003127908064277], ...","[[0.6536022972246243], [0.7333795803124573], [...","[[0.4650203316620493], [0.4073569980939774], [...","[[0.10691229404306314], [0.12635890217406154],...","[[0.05122301315681452], [0.0569182097563926], ...","[[0.16220538385027364], [0.1583851773203997], ...","[[0.7949495780540065], [0.7616877086144606], [..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,DumpSorting,"[[0.4443740108602035], [0.4198554271522224], [...","[[0.11714379292696687], [0.11891123876598562],...","[[0.03881323741764504], [0.05153303755877407],...","[[0.08762519892091923], [0.06520297724826178],...","[[0.6906060261445305], [0.6599746929087467], [...","[[0.14179914316112374], [0.1351493409367238], ...","[[0.19139161448942335], [0.17348673113547494],...","[[0.07422284713884825], [0.09670352108465813],...","[[0.1317324779677769], [0.09020753250794743], ...","[[0.5586159941850517], [0.5108376056341116], [..."
1068,71,0.435651,BinomialCoefficient,"[[0.17285142449268165], [0.10379548942729463],...","[[0.20717493665861564], [0.17033232915780083],...","[[0.0808215910630472], [0.09849913927409563], ...","[[0.1473923520875673], [0.1604632323470267], [...","[[0.6187939065384642], [0.5480429893580788], [...","[[0.3019095493624292], [0.27296925212631584], ...","[[0.16978911795791105], [0.14358128437411732],...","[[0.07202331112700304], [0.05879868796118298],...","[[0.17891189440486968], [0.13525945888672627],...","[[0.726855

In [17]:
df_brain_waves_skilled = pd.merge(df_brain_waves, df_skilled[['Participant', 'SkillLevel']], on='Participant', how='left')
df_brain_waves_skilled = df_brain_waves_skilled[["Participant", "SkillScore", "SkillLevel", "Algorithm", "Alpha", "Beta", "Gamma", "Theta", "Range4to50Hz","BaselineAlpha", "BaselineBeta", "BaselineGamma", "BaselineTheta", "BaselineRange4to50Hz"]]
df_brain_waves_skilled.to_csv(result_path+"/BrainWavesSkilled.csv")
df_brain_waves_skilled

,Participant,SkillScore,SkillLevel,Algorithm,Alpha,Beta,Gamma,Theta,Range4to50Hz,BaselineAlpha,BaselineBeta,BaselineGamma,BaselineTheta,BaselineRange4to50Hz
0,1,0.331385,Intermediate,IsPrime,"[[0.1778556627649099], [0.1294300454494536], [...","[[0.12785309461468938], [0.17481799804998913],...","[[0.07321136743334146], [0.08930578645043821],...","[[0.22873350091444938], [0.1873401960575567], ...","[[0.6076536257273902], [0.5808940260074379], [...","[[0.12893736631902714], [0.13052248391744597],...","[[0.1949327305938576], [0.2207488607445728], [...","[[0.05891106149513782], [0.10684924675674091],...","[[0.10238700518909745], [0.14285755578310447],...","[[0.4934974808769842], [0.605755140731115], [0..."
1,1,0.331385,Intermediate,SiebDesEratosthenes,"[[0.14399128899708588], [0.14083532302932217],...","[[0.17098830181532818], [0.18959792215957177],...","[[0.09960026421162532], [0.06637512989532202],...","[[0.09127189594337842], [0.16889922549878147],...","[[0.5074623755120865], [0.5770144398756503], [...","[[0.2973188375069371], [0.3396716104122783], [...","[[0.1334035676382684], [0.11980772650574263], ...","[[0.0678748617080782], [0.0716394708637403], [...","[[0.1019550621165257], [0.10428815863177555], ...","[[0.6333511677018893], [0.6724246817175873], [..."
2,1,0.331385,Intermediate,IsAnagram,"[[0.1295553937970955], [0.09282823877611668], ...","[[0.16554861150430653], [0.15001045660538426],...","[[0.07485087416136596], [0.08269131237739936],...","[[0.19058703508450428], [0.2601196234617058], ...","[[0.5673785866341523], [0.6079983141209369], [...","[[0.39693865402301315], [0.33341281436728526],...","[[0.13301306960007464], [0.18685213113646118],...","[[0.047830021245293564], [0.07456727192237615]...","[[0.13586575073301865], [0.08082012108699872],...","[[0.7335055798110417], [0.6977508092089181], [..."
3,1,0.331385,Intermediate,RemoveDoubleChar,"[[0.17094104741567773], [0.15124354622812303],...","[[0.1268823216723157], [0.159300180041804], [0...","[[0.061578963694041575], [0.05898304983434938]...","[[0.24139148749165498], [0.2645480523689333], ...","[[0.6276414818034795], [0.6662657112007767], [...","[[0.3656780614728621], [0.2924782015564169], [...","[[0.12628383741052132], [0.16834485494551116],...","[[0.06306193457649979], [0.08659003328539422],...","[[0.08198915510632611], [0.06843162002288498],...","[[0.6663144321737743], [0.6223935196646638], [..."
4,1,0.331385,Intermediate,BinToDecimal,"[[0.17921206709946502], [0.1407427079432058], ...","[[0.16396297808006471], [0.20131184220965245],...","[[0.1155854466642323], [0.12052159022409228], ...","[[0.1731292208812361], [0.25003127908064277], ...","[[0.6536022972246243], [0.7333795803124573], [...","[[0.4650203316620493], [0.4073569980939774], [...","[[0.10691229404306314], [0.12635890217406154],...","[[0.05122301315681452], [0.0569182097563926], ...","[[0.16220538385027364], [0.1583851773203997], ...","[[0.7949495780540065], [0.7616877086144606], [..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,"[[0.4443740108602035], [0.4198554271522224], [...","[[0.11714379292696687], [0.11891123876598562],...","[[0.03881323741764504], [0.05153303755877407],...","[[0.08762519892091923], [0.06520297724826178],...","[[0.6906060261445305], [0.6599746929087467], [...","[[0.14179914316112374], [0.1351493409367238], ...","[[0.19139161448942335], [0.17348673113547494],...","[[0.07422284713884825], [0.09670352108465813],...","[[0.1317324779677769], [0.09020753250794743], ...","[[0.5586159941850517], [0.5108376056341116], [..."
1068,71,0.435651,Expert,BinomialCoefficient,"[[0.17285142449268165], [0.10379548942729463],...","[[0.20717493665861564], [0.17033232915780083],...","[[0.0808215910630472], [0.09849913927409563], ...","[[0.1473923520875673], [0.1604632323470267], [...","[[0.6187939065384642], [0.5480429893580788], [...","[[0.3019095493624292], [0.27296925212631584], ...","[[0.16978911795791105], [0.14358128437411732],...","[[0.0720233111270030

In [18]:
df_brain_waves_skilled_unique = df_brain_waves_skilled[['Participant', 'SkillLevel']]
df_brain_waves_skilled_unique = df_brain_waves_skilled_unique.drop_duplicates()
df_brain_waves_skilled_unique.groupby('SkillLevel')['Participant'].count()

SkillLevel
Expert          13
Intermediate    12
Novice          12
Name: Participant, dtype: int64

In [19]:
# Statistical EEG features
def get_freqband_features(data):
    mean = np.mean(data)
    std_dev = np.std(data)
    skewness = skew(data)
    kurt = kurtosis(data)
    variance = np.var(data)
    median = np.median(data)    
    zero_cross_rate = len(find_peaks(data)[0]) / (len(data) - 1)
    entropy_data = entropy(data)
    
    return mean, std_dev, skewness, kurt, variance, median, zero_cross_rate, entropy_data

# Features

# 1. Based on Algorithm (ignoring electrode positions)

## 1.1. Brain Waves --> Alpha, Beta, Gamma, Theta waves

### 1.1.1 Code Comprehension Data

In [20]:
#get the code comprehension data brain waves features
df_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore', 'SkillLevel', 'Algorithm', 
            'Alpha_Mean', 'Beta_Mean', 'Gamma_Mean', 'Theta_Mean',
            'Alpha_StdDev', 'Beta_StdDev', 'Gamma_StdDev', 'Theta_StdDev',
            'Alpha_Skewness', 'Beta_Skewness', 'Gamma_Skewness', 'Theta_Skewness',
            'Alpha_Kurtosis', 'Beta_Kurtosis', 'Gamma_Kurtosis', 'Theta_Kurtosis',
            'Alpha_Variance', 'Beta_Variance', 'Gamma_Variance', 'Theta_Variance',
            'Alpha_Median', 'Beta_Median', 'Gamma_Median', 'Theta_Median',
            'Alpha_ZeroCrossRate', 'Beta_ZeroCrossRate', 'Gamma_ZeroCrossRate', 'Theta_ZeroCrossRate',
            'Alpha_Entropy', 'Beta_Entropy', 'Gamma_Entropy', 'Theta_Entropy'])

for participant in tqdm(df_brain_waves_skilled["Participant"].unique(), total=len(df_brain_waves_skilled["Participant"].unique())):
    df_participant = df_brain_waves_skilled[df_brain_waves_skilled["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    algorithm = df_participant["Algorithm"]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_alpha = np.zeros(shape=(64, 0))
    merged_beta = np.zeros(shape=(64, 0))
    merged_gamma = np.zeros(shape=(64, 0))
    merged_theta = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        alpha = row["Alpha"]
        beta = row["Beta"]
        gamma = row["Gamma"]
        theta = row["Theta"]

        

        merged_alpha = np.concatenate((merged_alpha, alpha), axis=1)
        merged_beta = np.concatenate((merged_beta, beta), axis=1)
        merged_gamma = np.concatenate((merged_gamma, gamma), axis=1)
        merged_theta = np.concatenate((merged_theta, theta), axis=1)
       
    # Calculate statistical features for each algorithm
    for idx, algorithm in enumerate(algorithm):
        alpha_channeled = merged_alpha[idx, :]
        beta_channeled = merged_beta[idx, :]
        gamma_channeled = merged_gamma[idx, :]
        theta_channeled = merged_theta[idx, :]

        
        # Use the calculate_channel_statistics function on each frequency band
        alpha_stats = get_freqband_features(alpha_channeled)
        beta_stats = get_freqband_features(beta_channeled)
        gamma_stats = get_freqband_features(gamma_channeled)
        theta_stats = get_freqband_features(theta_channeled)
        # Create a DataFrame for the current algorithm
        algorithm_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Algorithm': algorithm,
            'Alpha_Mean': alpha_stats[0],
            'Beta_Mean': beta_stats[0],
            'Gamma_Mean': gamma_stats[0],
            'Theta_Mean': theta_stats[0],
            'Alpha_StdDev': alpha_stats[1],
            'Beta_StdDev': beta_stats[1],
            'Gamma_StdDev': gamma_stats[1],
            'Theta_StdDev': theta_stats[1],
            'Alpha_Skewness': alpha_stats[2],
            'Beta_Skewness': beta_stats[2],
            'Gamma_Skewness': gamma_stats[2],
            'Theta_Skewness': theta_stats[2],
            'Alpha_Kurtosis': alpha_stats[3],
            'Beta_Kurtosis': beta_stats[3],
            'Gamma_Kurtosis': gamma_stats[3],
            'Theta_Kurtosis': theta_stats[3],
            'Alpha_Variance': alpha_stats[4],
            'Beta_Variance': beta_stats[4],
            'Gamma_Variance': gamma_stats[4],
            'Theta_Variance': theta_stats[4],
            'Alpha_Median': alpha_stats[5],
            'Beta_Median': beta_stats[5],
            'Gamma_Median': gamma_stats[5],
            'Theta_Median': theta_stats[5],
            'Alpha_ZeroCrossRate': alpha_stats[6],
            'Beta_ZeroCrossRate': beta_stats[6],
            'Gamma_ZeroCrossRate': gamma_stats[6],
            'Theta_ZeroCrossRate': theta_stats[6],
            'Alpha_Entropy': alpha_stats[7],
            'Beta_Entropy': beta_stats[7],
            'Gamma_Entropy': gamma_stats[7],
            'Theta_Entropy': theta_stats[7],
        }, index=[0])

        df_brain_waves_stats = pd.concat([df_brain_waves_stats, algorithm_df], ignore_index=True)
        
df_brain_waves_stats.to_csv(result_path + "/features_alg_abgtbw_ccdata.csv")
df_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\1344662213.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_brain_waves_stats = pd.concat([df_brain_waves_stats, algorithm_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Algorithm,Alpha_Mean,Beta_Mean,Gamma_Mean,Theta_Mean,Alpha_StdDev,Beta_StdDev,...,Gamma_Median,Theta_Median,Alpha_ZeroCrossRate,Beta_ZeroCrossRate,Gamma_ZeroCrossRate,Theta_ZeroCrossRate,Alpha_Entropy,Beta_Entropy,Gamma_Entropy,Theta_Entropy
0,1,0.331385,Intermediate,IsPrime,0.166390,0.150510,0.090489,0.157715,0.047104,0.025264,...,0.086674,0.166295,0.290323,0.354839,0.322581,0.354839,3.423196,3.451393,3.450221,3.421571
1,1,0.331385,Intermediate,SiebDesEratosthenes,0.162938,0.180551,0.085663,0.183140,0.047777,0.028309,...,0.087033,0.173035,0.322581,0.322581,0.354839,0.290323,3.422616,3.453243,3.444997,3.435099
2,1,0.331385,Intermediate,IsAnagram,0.137152,0.190865,0.095615,0.166988,0.047963,0.035197,...,0.093597,0.154126,0.290323,0.354839,0.322581,0.387097,3.403825,3.448273,3.436679,3.412071
3,1,0.331385,Intermediate,RemoveDoubleChar,0.171606,0.141907,0.056353,0.227676,0.053040,0.035415,...,0.057599,0.206876,0.290323,0.322581,0.290323,0.354839,3.418012,3.434288,3.439739,3.415109
4,1,0.331385,Intermediate,BinToDecimal,0.184896,0.130419,0.052744,0.244464,0.065693,0.028818,...,0.050492,0.230446,0.322581,0.354839,0.322581,0.354839,3.404437,3.441623,3.419171,3.418192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,0.129026,0.170598,0.094394,0.147180,0.054072,0.034282,...,0.093606,0.122978,0.258065,0.419355,0.258065,0.290323,3.376887,3.445145,3.407616,3.331527
1068,71,0.435651,Expert,BinomialCoefficient,0.150430,0.156127,0.054409,0.228501,0.045355,0.030921,...,0.047544,0.221687,0.354839,0.322581,0.258065,0.322581,3.418642,3.446684,3.409007,3.411158
1069,71,0.435651,Expert,IsAnagram,0.147308,0.151950,0.056556,0.229323,0.043776,0.029501,...,0.049660,0.217633,0.354839,0.290323,0.290323,0.354839,3.419682,3.447171,3.409741,3.415282
1070,71,0.435651,Expert,ArrayAverage,0.120768,0.156797,0.082784,0.200146,0.038786,0.030899,...,0.077092,0.202294,0.290323,0.354839,0.290323,0.322581,3.410059,3.446216,3.430600,3.399013


### 1.1.2 Baseline Data

In [21]:
df_brain_waves= df_brain_waves_skilled.copy()

df_baseline_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore', 'SkillLevel','Algorithm', 
            'Alpha_Mean', 'Beta_Mean', 'Gamma_Mean', 'Theta_Mean',
            'Alpha_StdDev', 'Beta_StdDev', 'Gamma_StdDev', 'Theta_StdDev',
            'Alpha_Skewness', 'Beta_Skewness', 'Gamma_Skewness', 'Theta_Skewness',
            'Alpha_Kurtosis', 'Beta_Kurtosis', 'Gamma_Kurtosis', 'Theta_Kurtosis',
            'Alpha_Variance', 'Beta_Variance', 'Gamma_Variance', 'Theta_Variance',
            'Alpha_Median', 'Beta_Median', 'Gamma_Median', 'Theta_Median',
            'Alpha_ZeroCrossRate', 'Beta_ZeroCrossRate', 'Gamma_ZeroCrossRate', 'Theta_ZeroCrossRate',
            'Alpha_Entropy', 'Beta_Entropy', 'Gamma_Entropy', 'Theta_Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    algorithm = df_participant["Algorithm"]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_alpha = np.zeros(shape=(64, 0))
    merged_beta = np.zeros(shape=(64, 0))
    merged_gamma = np.zeros(shape=(64, 0))
    merged_theta = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        alpha = row["BaselineAlpha"]
        beta = row["BaselineBeta"]
        gamma = row["BaselineGamma"]
        theta = row["BaselineTheta"]

        

        merged_alpha = np.concatenate((merged_alpha, alpha), axis=1)
        merged_beta = np.concatenate((merged_beta, beta), axis=1)
        merged_gamma = np.concatenate((merged_gamma, gamma), axis=1)
        merged_theta = np.concatenate((merged_theta, theta), axis=1)
       
    # Calculate statistical features for each algorithm
    for idx, algorithm in enumerate(algorithm):
        alpha_channeled = merged_alpha[idx, :]
        beta_channeled = merged_beta[idx, :]
        gamma_channeled = merged_gamma[idx, :]
        theta_channeled = merged_theta[idx, :]

        
        # Use the calculate_channel_statistics function on each frequency band
        alpha_stats = get_freqband_features(alpha_channeled)
        beta_stats = get_freqband_features(beta_channeled)
        gamma_stats = get_freqband_features(gamma_channeled)
        theta_stats = get_freqband_features(theta_channeled)
        # Create a DataFrame for the current algorithm
        algorithm_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'Algorithm': algorithm,
            'SkillLevel': skill_level,
            'Alpha_Mean': alpha_stats[0],
            'Beta_Mean': beta_stats[0],
            'Gamma_Mean': gamma_stats[0],
            'Theta_Mean': theta_stats[0],
            'Alpha_StdDev': alpha_stats[1],
            'Beta_StdDev': beta_stats[1],
            'Gamma_StdDev': gamma_stats[1],
            'Theta_StdDev': theta_stats[1],
            'Alpha_Skewness': alpha_stats[2],
            'Beta_Skewness': beta_stats[2],
            'Gamma_Skewness': gamma_stats[2],
            'Theta_Skewness': theta_stats[2],
            'Alpha_Kurtosis': alpha_stats[3],
            'Beta_Kurtosis': beta_stats[3],
            'Gamma_Kurtosis': gamma_stats[3],
            'Theta_Kurtosis': theta_stats[3],
            'Alpha_Variance': alpha_stats[4],
            'Beta_Variance': beta_stats[4],
            'Gamma_Variance': gamma_stats[4],
            'Theta_Variance': theta_stats[4],
            'Alpha_Median': alpha_stats[5],
            'Beta_Median': beta_stats[5],
            'Gamma_Median': gamma_stats[5],
            'Theta_Median': theta_stats[5],
            'Alpha_ZeroCrossRate': alpha_stats[6],
            'Beta_ZeroCrossRate': beta_stats[6],
            'Gamma_ZeroCrossRate': gamma_stats[6],
            'Theta_ZeroCrossRate': theta_stats[6],
            'Alpha_Entropy': alpha_stats[7],
            'Beta_Entropy': beta_stats[7],
            'Gamma_Entropy': gamma_stats[7],
            'Theta_Entropy': theta_stats[7],
        }, index=[0])

        df_baseline_brain_waves_stats = pd.concat([df_baseline_brain_waves_stats, algorithm_df], ignore_index=True)

df_baseline_brain_waves_stats.to_csv(result_path+'/features_alg_abgtbw_bldata.csv')
df_baseline_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\1589296233.py:91: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_baseline_brain_waves_stats = pd.concat([df_baseline_brain_waves_stats, algorithm_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Algorithm,Alpha_Mean,Beta_Mean,Gamma_Mean,Theta_Mean,Alpha_StdDev,Beta_StdDev,...,Gamma_Median,Theta_Median,Alpha_ZeroCrossRate,Beta_ZeroCrossRate,Gamma_ZeroCrossRate,Theta_ZeroCrossRate,Alpha_Entropy,Beta_Entropy,Gamma_Entropy,Theta_Entropy
0,1,0.331385,Intermediate,IsPrime,0.395663,0.113486,0.063130,0.097166,0.169601,0.034206,...,0.057804,0.099865,0.354839,0.258065,0.290323,0.322581,3.373729,3.420319,3.412292,3.380473
1,1,0.331385,Intermediate,SiebDesEratosthenes,0.369195,0.144116,0.063106,0.129108,0.173634,0.039341,...,0.060817,0.125693,0.322581,0.258065,0.258065,0.290323,3.363019,3.427183,3.409783,3.383050
2,1,0.331385,Intermediate,IsAnagram,0.292649,0.153357,0.077229,0.130586,0.114222,0.036661,...,0.076072,0.132598,0.322581,0.258065,0.322581,0.258065,3.386727,3.437070,3.449016,3.417734
3,1,0.331385,Intermediate,RemoveDoubleChar,0.325351,0.135161,0.055520,0.159799,0.129947,0.033203,...,0.057870,0.152070,0.322581,0.322581,0.354839,0.354839,3.388634,3.434490,3.420064,3.403502
4,1,0.331385,Intermediate,BinToDecimal,0.330490,0.129424,0.048896,0.177055,0.128879,0.030626,...,0.048574,0.174883,0.322581,0.290323,0.290323,0.290323,3.392261,3.435388,3.417281,3.407822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,0.126629,0.254843,0.098818,0.102414,0.039961,0.067925,...,0.089098,0.088528,0.290323,0.290323,0.258065,0.290323,3.415599,3.431161,3.388070,3.370472
1068,71,0.435651,Expert,BinomialCoefficient,0.201077,0.238843,0.061440,0.132362,0.056037,0.063251,...,0.054584,0.133069,0.322581,0.290323,0.354839,0.290323,3.426420,3.430424,3.371645,3.427233
1069,71,0.435651,Expert,IsAnagram,0.196164,0.238398,0.062414,0.133639,0.053574,0.062597,...,0.052647,0.138148,0.354839,0.322581,0.354839,0.290323,3.428545,3.430720,3.369070,3.421898
1070,71,0.435651,Expert,ArrayAverage,0.161211,0.237086,0.085933,0.120501,0.046872,0.051356,...,0.083773,0.118976,0.322581,0.322581,0.322581,0.387097,3.423258,3.442386,3.413989,3.435593


## 1.2 Brain Waves --> 4 to 50 Hz range 

In [22]:
df_brain_waves= df_brain_waves_skilled.copy()

### 1.2.1 Code Comprehension Data

In [23]:

df_4to50Hz_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore','SkillLevel', 'Algorithm', 
            'Mean', 'StdDev', 'Skewness', 'Kurtosis', 'Variance', 'ZeroCrossRate', 'Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    algorithm = df_participant["Algorithm"]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_freq_range = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        freq_range = row["Range4to50Hz"]
        merged_freq_range = np.concatenate((merged_freq_range, freq_range), axis=1)
       
    # Calculate statistical features for each algorithm
    for idx, algorithm in enumerate(algorithm):
        channeled = merged_freq_range[idx, :]
       
        
        # Use the calculate_channel_statistics function on frequency band
        stats = get_freqband_features(channeled)
        # Create a DataFrame for the current algorithm
        algorithm_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Algorithm': algorithm,
            'Mean': stats[0],
            'StdDev': stats[1],
            'Skewness': stats[2],
            'Kurtosis': stats[3],
            'Variance': stats[4],
            'Median': stats[5],
            'ZeroCrossRate': stats[6],
            'Entropy': stats[7],
        }, index=[0])

        df_4to50Hz_brain_waves_stats = pd.concat([df_4to50Hz_brain_waves_stats, algorithm_df], ignore_index=True)

df_4to50Hz_brain_waves_stats.to_csv(result_path+'/features_alg_4to50hzbw_ccdata.csv')
df_4to50Hz_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\2057372047.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_4to50Hz_brain_waves_stats = pd.concat([df_4to50Hz_brain_waves_stats, algorithm_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Algorithm,Mean,StdDev,Skewness,Kurtosis,Variance,ZeroCrossRate,Entropy,Median
0,1,0.331385,Intermediate,IsPrime,0.578004,0.060779,0.187927,-0.328752,0.003694,0.290323,3.460217,0.574458
1,1,0.331385,Intermediate,SiebDesEratosthenes,0.630075,0.057554,-0.270284,-0.113730,0.003312,0.322581,3.461512,0.629625
2,1,0.331385,Intermediate,IsAnagram,0.603259,0.069906,0.057028,-0.523495,0.004887,0.258065,3.458999,0.596230
3,1,0.331385,Intermediate,RemoveDoubleChar,0.610693,0.090608,-0.509353,-0.256388,0.008210,0.290323,3.454321,0.609872
4,1,0.331385,Intermediate,BinToDecimal,0.625720,0.094338,-0.477489,-0.355320,0.008900,0.258065,3.453964,0.628981
...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,0.552769,0.101330,-0.154958,0.116930,0.010268,0.290323,3.448439,0.546036
1068,71,0.435651,Expert,BinomialCoefficient,0.605928,0.093571,-0.033833,-0.604552,0.008755,0.354839,3.453671,0.602918
1069,71,0.435651,Expert,IsAnagram,0.601816,0.092050,-0.030185,-0.724780,0.008473,0.322581,3.453911,0.597933
1070,71,0.435651,Expert,ArrayAverage,0.574399,0.075900,0.340126,-0.232116,0.005761,0.354839,3.457070,0.578466


### 1.2.2 Baseline Data

In [24]:
df_brain_waves= df_brain_waves_skilled.copy()
df_brain_waves

,Participant,SkillScore,SkillLevel,Algorithm,Alpha,Beta,Gamma,Theta,Range4to50Hz,BaselineAlpha,BaselineBeta,BaselineGamma,BaselineTheta,BaselineRange4to50Hz
0,1,0.331385,Intermediate,IsPrime,"[[0.1778556627649099], [0.1294300454494536], [...","[[0.12785309461468938], [0.17481799804998913],...","[[0.07321136743334146], [0.08930578645043821],...","[[0.22873350091444938], [0.1873401960575567], ...","[[0.6076536257273902], [0.5808940260074379], [...","[[0.12893736631902714], [0.13052248391744597],...","[[0.1949327305938576], [0.2207488607445728], [...","[[0.05891106149513782], [0.10684924675674091],...","[[0.10238700518909745], [0.14285755578310447],...","[[0.4934974808769842], [0.605755140731115], [0..."
1,1,0.331385,Intermediate,SiebDesEratosthenes,"[[0.14399128899708588], [0.14083532302932217],...","[[0.17098830181532818], [0.18959792215957177],...","[[0.09960026421162532], [0.06637512989532202],...","[[0.09127189594337842], [0.16889922549878147],...","[[0.5074623755120865], [0.5770144398756503], [...","[[0.2973188375069371], [0.3396716104122783], [...","[[0.1334035676382684], [0.11980772650574263], ...","[[0.0678748617080782], [0.0716394708637403], [...","[[0.1019550621165257], [0.10428815863177555], ...","[[0.6333511677018893], [0.6724246817175873], [..."
2,1,0.331385,Intermediate,IsAnagram,"[[0.1295553937970955], [0.09282823877611668], ...","[[0.16554861150430653], [0.15001045660538426],...","[[0.07485087416136596], [0.08269131237739936],...","[[0.19058703508450428], [0.2601196234617058], ...","[[0.5673785866341523], [0.6079983141209369], [...","[[0.39693865402301315], [0.33341281436728526],...","[[0.13301306960007464], [0.18685213113646118],...","[[0.047830021245293564], [0.07456727192237615]...","[[0.13586575073301865], [0.08082012108699872],...","[[0.7335055798110417], [0.6977508092089181], [..."
3,1,0.331385,Intermediate,RemoveDoubleChar,"[[0.17094104741567773], [0.15124354622812303],...","[[0.1268823216723157], [0.159300180041804], [0...","[[0.061578963694041575], [0.05898304983434938]...","[[0.24139148749165498], [0.2645480523689333], ...","[[0.6276414818034795], [0.6662657112007767], [...","[[0.3656780614728621], [0.2924782015564169], [...","[[0.12628383741052132], [0.16834485494551116],...","[[0.06306193457649979], [0.08659003328539422],...","[[0.08198915510632611], [0.06843162002288498],...","[[0.6663144321737743], [0.6223935196646638], [..."
4,1,0.331385,Intermediate,BinToDecimal,"[[0.17921206709946502], [0.1407427079432058], ...","[[0.16396297808006471], [0.20131184220965245],...","[[0.1155854466642323], [0.12052159022409228], ...","[[0.1731292208812361], [0.25003127908064277], ...","[[0.6536022972246243], [0.7333795803124573], [...","[[0.4650203316620493], [0.4073569980939774], [...","[[0.10691229404306314], [0.12635890217406154],...","[[0.05122301315681452], [0.0569182097563926], ...","[[0.16220538385027364], [0.1583851773203997], ...","[[0.7949495780540065], [0.7616877086144606], [..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,"[[0.4443740108602035], [0.4198554271522224], [...","[[0.11714379292696687], [0.11891123876598562],...","[[0.03881323741764504], [0.05153303755877407],...","[[0.08762519892091923], [0.06520297724826178],...","[[0.6906060261445305], [0.6599746929087467], [...","[[0.14179914316112374], [0.1351493409367238], ...","[[0.19139161448942335], [0.17348673113547494],...","[[0.07422284713884825], [0.09670352108465813],...","[[0.1317324779677769], [0.09020753250794743], ...","[[0.5586159941850517], [0.5108376056341116], [..."
1068,71,0.435651,Expert,BinomialCoefficient,"[[0.17285142449268165], [0.10379548942729463],...","[[0.20717493665861564], [0.17033232915780083],...","[[0.0808215910630472], [0.09849913927409563], ...","[[0.1473923520875673], [0.1604632323470267], [...","[[0.6187939065384642], [0.5480429893580788], [...","[[0.3019095493624292], [0.27296925212631584], ...","[[0.16978911795791105], [0.14358128437411732],...","[[0.0720233111270030

In [25]:

df_bl_4to50Hz_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore','SkillLevel', 'Algorithm', 
            'Mean', 'StdDev', 'Skewness', 'Kurtosis', 'Variance', 'ZeroCrossRate', 'Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    algorithm = df_participant["Algorithm"]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_freq_range = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        freq_range = row["BaselineRange4to50Hz"]
        merged_freq_range = np.concatenate((merged_freq_range, freq_range), axis=1)
       
    # Calculate statistical features for each algorithm
    for idx, algorithm in enumerate(algorithm):
        channeled = merged_freq_range[idx, :]
       
        
        # Use the calculate_channel_statistics function on frequency band
        stats = get_freqband_features(channeled)
        # Create a DataFrame for the current algorithm
        algorithm_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Algorithm': algorithm,
            'Mean': stats[0],
            'StdDev': stats[1],
            'Skewness': stats[2],
            'Kurtosis': stats[3],
            'Variance': stats[4],
            'Median': stats[5],
            'ZeroCrossRate': stats[6],
            'Entropy': stats[7],
        }, index=[0])

        df_bl_4to50Hz_brain_waves_stats = pd.concat([df_bl_4to50Hz_brain_waves_stats, algorithm_df], ignore_index=True)
df_bl_4to50Hz_brain_waves_stats.to_csv(result_path+'/features_alg_4to50hzbw_bldata.csv')
df_bl_4to50Hz_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\3755910020.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_bl_4to50Hz_brain_waves_stats = pd.concat([df_bl_4to50Hz_brain_waves_stats, algorithm_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Algorithm,Mean,StdDev,Skewness,Kurtosis,Variance,ZeroCrossRate,Entropy,Median
0,1,0.331385,Intermediate,IsPrime,0.681020,0.108926,0.009201,-0.338011,0.011865,0.354839,3.452806,0.680778
1,1,0.331385,Intermediate,SiebDesEratosthenes,0.718634,0.098006,0.531828,0.046404,0.009605,0.322581,3.456580,0.700738
2,1,0.331385,Intermediate,IsAnagram,0.664859,0.072894,-0.240494,-0.108777,0.005314,0.387097,3.459637,0.679650
3,1,0.331385,Intermediate,RemoveDoubleChar,0.688845,0.085296,-0.231119,-0.574746,0.007275,0.290323,3.457946,0.707573
4,1,0.331385,Intermediate,BinToDecimal,0.700076,0.087189,-0.392213,-0.166185,0.007602,0.225806,3.457792,0.713150
...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,0.595582,0.091360,0.032683,-0.810736,0.008347,0.322581,3.453887,0.596684
1068,71,0.435651,Expert,BinomialCoefficient,0.649854,0.087392,-0.231387,-1.396360,0.007637,0.290323,3.456554,0.665943
1069,71,0.435651,Expert,IsAnagram,0.646340,0.085891,-0.240050,-1.296106,0.007377,0.290323,3.456765,0.660856
1070,71,0.435651,Expert,ArrayAverage,0.621977,0.067725,-0.328876,-0.585665,0.004587,0.354839,3.459706,0.634634


# 2. Based on Electrode Position

In [26]:
df_brain_waves= df_brain_waves_skilled.copy()
df_brain_waves


,Participant,SkillScore,SkillLevel,Algorithm,Alpha,Beta,Gamma,Theta,Range4to50Hz,BaselineAlpha,BaselineBeta,BaselineGamma,BaselineTheta,BaselineRange4to50Hz
0,1,0.331385,Intermediate,IsPrime,"[[0.1778556627649099], [0.1294300454494536], [...","[[0.12785309461468938], [0.17481799804998913],...","[[0.07321136743334146], [0.08930578645043821],...","[[0.22873350091444938], [0.1873401960575567], ...","[[0.6076536257273902], [0.5808940260074379], [...","[[0.12893736631902714], [0.13052248391744597],...","[[0.1949327305938576], [0.2207488607445728], [...","[[0.05891106149513782], [0.10684924675674091],...","[[0.10238700518909745], [0.14285755578310447],...","[[0.4934974808769842], [0.605755140731115], [0..."
1,1,0.331385,Intermediate,SiebDesEratosthenes,"[[0.14399128899708588], [0.14083532302932217],...","[[0.17098830181532818], [0.18959792215957177],...","[[0.09960026421162532], [0.06637512989532202],...","[[0.09127189594337842], [0.16889922549878147],...","[[0.5074623755120865], [0.5770144398756503], [...","[[0.2973188375069371], [0.3396716104122783], [...","[[0.1334035676382684], [0.11980772650574263], ...","[[0.0678748617080782], [0.0716394708637403], [...","[[0.1019550621165257], [0.10428815863177555], ...","[[0.6333511677018893], [0.6724246817175873], [..."
2,1,0.331385,Intermediate,IsAnagram,"[[0.1295553937970955], [0.09282823877611668], ...","[[0.16554861150430653], [0.15001045660538426],...","[[0.07485087416136596], [0.08269131237739936],...","[[0.19058703508450428], [0.2601196234617058], ...","[[0.5673785866341523], [0.6079983141209369], [...","[[0.39693865402301315], [0.33341281436728526],...","[[0.13301306960007464], [0.18685213113646118],...","[[0.047830021245293564], [0.07456727192237615]...","[[0.13586575073301865], [0.08082012108699872],...","[[0.7335055798110417], [0.6977508092089181], [..."
3,1,0.331385,Intermediate,RemoveDoubleChar,"[[0.17094104741567773], [0.15124354622812303],...","[[0.1268823216723157], [0.159300180041804], [0...","[[0.061578963694041575], [0.05898304983434938]...","[[0.24139148749165498], [0.2645480523689333], ...","[[0.6276414818034795], [0.6662657112007767], [...","[[0.3656780614728621], [0.2924782015564169], [...","[[0.12628383741052132], [0.16834485494551116],...","[[0.06306193457649979], [0.08659003328539422],...","[[0.08198915510632611], [0.06843162002288498],...","[[0.6663144321737743], [0.6223935196646638], [..."
4,1,0.331385,Intermediate,BinToDecimal,"[[0.17921206709946502], [0.1407427079432058], ...","[[0.16396297808006471], [0.20131184220965245],...","[[0.1155854466642323], [0.12052159022409228], ...","[[0.1731292208812361], [0.25003127908064277], ...","[[0.6536022972246243], [0.7333795803124573], [...","[[0.4650203316620493], [0.4073569980939774], [...","[[0.10691229404306314], [0.12635890217406154],...","[[0.05122301315681452], [0.0569182097563926], ...","[[0.16220538385027364], [0.1583851773203997], ...","[[0.7949495780540065], [0.7616877086144606], [..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,71,0.435651,Expert,DumpSorting,"[[0.4443740108602035], [0.4198554271522224], [...","[[0.11714379292696687], [0.11891123876598562],...","[[0.03881323741764504], [0.05153303755877407],...","[[0.08762519892091923], [0.06520297724826178],...","[[0.6906060261445305], [0.6599746929087467], [...","[[0.14179914316112374], [0.1351493409367238], ...","[[0.19139161448942335], [0.17348673113547494],...","[[0.07422284713884825], [0.09670352108465813],...","[[0.1317324779677769], [0.09020753250794743], ...","[[0.5586159941850517], [0.5108376056341116], [..."
1068,71,0.435651,Expert,BinomialCoefficient,"[[0.17285142449268165], [0.10379548942729463],...","[[0.20717493665861564], [0.17033232915780083],...","[[0.0808215910630472], [0.09849913927409563], ...","[[0.1473923520875673], [0.1604632323470267], [...","[[0.6187939065384642], [0.5480429893580788], [...","[[0.3019095493624292], [0.27296925212631584], ...","[[0.16978911795791105], [0.14358128437411732],...","[[0.0720233111270030

In [27]:
# read in one example file
example_cross_file = df_eeg_data.iloc[0]["CrossEEG"]
# read in the cross file
cross_eeg_data = mne.io.read_raw_fif(example_cross_file, verbose='ERROR')
# get the channel names
channel_names = cross_eeg_data.ch_names

## 2.1 Brain Waves --> Alpha, Beta, Gamma, Theta waves

### 2.1.1 Code Comprehension Data

In [28]:
#get features for code comprehension data
df_elec_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore','SkillLevel', 'Channel', 
            'Alpha_Mean', 'Beta_Mean', 'Gamma_Mean', 'Theta_Mean',
            'Alpha_StdDev', 'Beta_StdDev', 'Gamma_StdDev', 'Theta_StdDev',
            'Alpha_Skewness', 'Beta_Skewness', 'Gamma_Skewness', 'Theta_Skewness',
            'Alpha_Kurtosis', 'Beta_Kurtosis', 'Gamma_Kurtosis', 'Theta_Kurtosis',
            'Alpha_Variance', 'Beta_Variance', 'Gamma_Variance', 'Theta_Variance',
            'Alpha_Median', 'Beta_Median', 'Gamma_Median', 'Theta_Median',
            'Alpha_ZeroCrossRate', 'Beta_ZeroCrossRate', 'Gamma_ZeroCrossRate', 'Theta_ZeroCrossRate',
            'Alpha_Entropy', 'Beta_Entropy', 'Gamma_Entropy', 'Theta_Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_alpha = np.zeros(shape=(64, 0))
    merged_beta = np.zeros(shape=(64, 0))
    merged_gamma = np.zeros(shape=(64, 0))
    merged_theta = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        alpha = row["Alpha"]
        beta = row["Beta"]
        gamma = row["Gamma"]
        theta = row["Theta"]

        merged_alpha = np.concatenate((merged_alpha, alpha), axis=1)
        merged_beta = np.concatenate((merged_beta, beta), axis=1)
        merged_gamma = np.concatenate((merged_gamma, gamma), axis=1)
        merged_theta = np.concatenate((merged_theta, theta), axis=1)
    # Calculate statistical features for each channel
    for idx, channel in enumerate(channel_names):
        alpha_channeled = merged_alpha[idx, :]
        beta_channeled = merged_beta[idx, :]
        gamma_channeled = merged_gamma[idx, :]
        theta_channeled = merged_theta[idx, :]
        
        # Use the calculate_channel_statistics function on each frequency band
        alpha_stats = get_freqband_features(alpha_channeled)
        beta_stats = get_freqband_features(beta_channeled)
        gamma_stats = get_freqband_features(gamma_channeled)
        theta_stats = get_freqband_features(theta_channeled)

        
        
        # Create a DataFrame for the current channel
        channel_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Channel': channel,
            'Alpha_Mean': alpha_stats[0],
            'Beta_Mean': beta_stats[0],
            'Gamma_Mean': gamma_stats[0],
            'Theta_Mean': theta_stats[0],
            'Alpha_StdDev': alpha_stats[1],
            'Beta_StdDev': beta_stats[1],
            'Gamma_StdDev': gamma_stats[1],
            'Theta_StdDev': theta_stats[1],
            'Alpha_Skewness': alpha_stats[2],
            'Beta_Skewness': beta_stats[2],
            'Gamma_Skewness': gamma_stats[2],
            'Theta_Skewness': theta_stats[2],
            'Alpha_Kurtosis': alpha_stats[3],
            'Beta_Kurtosis': beta_stats[3],
            'Gamma_Kurtosis': gamma_stats[3],
            'Theta_Kurtosis': theta_stats[3],
            'Alpha_Variance': alpha_stats[4],
            'Beta_Variance': beta_stats[4],
            'Gamma_Variance': gamma_stats[4],
            'Theta_Variance': theta_stats[4],
            'Alpha_Median': alpha_stats[5],
            'Beta_Median': beta_stats[5],
            'Gamma_Median': gamma_stats[5],
            'Theta_Median': theta_stats[5],
            'Alpha_ZeroCrossRate': alpha_stats[6],
            'Beta_ZeroCrossRate': beta_stats[6],
            'Gamma_ZeroCrossRate': gamma_stats[6],
            'Theta_ZeroCrossRate': theta_stats[6],
            'Alpha_Entropy': alpha_stats[7],
            'Beta_Entropy': beta_stats[7],
            'Gamma_Entropy': gamma_stats[7],
            'Theta_Entropy': theta_stats[7]
        }, index=[0])

        df_elec_brain_waves_stats = pd.concat([df_elec_brain_waves_stats, channel_df], ignore_index=True)

df_elec_brain_waves_stats.to_csv(result_path+'/features_elec_abgtbw_ccdata.csv')
df_elec_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\1908312530.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_elec_brain_waves_stats = pd.concat([df_elec_brain_waves_stats, channel_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Channel,Alpha_Mean,Beta_Mean,Gamma_Mean,Theta_Mean,Alpha_StdDev,Beta_StdDev,...,Gamma_Median,Theta_Median,Alpha_ZeroCrossRate,Beta_ZeroCrossRate,Gamma_ZeroCrossRate,Theta_ZeroCrossRate,Alpha_Entropy,Beta_Entropy,Gamma_Entropy,Theta_Entropy
0,1,0.331385,Intermediate,Fp1,0.166390,0.150510,0.090489,0.157715,0.047104,0.025264,...,0.086674,0.166295,0.290323,0.354839,0.322581,0.354839,3.423196,3.451393,3.450221,3.421571
1,1,0.331385,Intermediate,Fp2,0.162938,0.180551,0.085663,0.183140,0.047777,0.028309,...,0.087033,0.173035,0.322581,0.322581,0.354839,0.290323,3.422616,3.453243,3.444997,3.435099
2,1,0.331385,Intermediate,F7,0.137152,0.190865,0.095615,0.166988,0.047963,0.035197,...,0.093597,0.154126,0.290323,0.354839,0.322581,0.387097,3.403825,3.448273,3.436679,3.412071
3,1,0.331385,Intermediate,F3,0.171606,0.141907,0.056353,0.227676,0.053040,0.035415,...,0.057599,0.206876,0.290323,0.322581,0.290323,0.354839,3.418012,3.434288,3.439739,3.415109
4,1,0.331385,Intermediate,Fz,0.184896,0.130419,0.052744,0.244464,0.065693,0.028818,...,0.050492,0.230446,0.322581,0.354839,0.322581,0.354839,3.404437,3.441623,3.419171,3.418192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2363,71,0.435651,Expert,PO7,0.216094,0.181458,0.039566,0.231026,0.062641,0.037594,...,0.035469,0.231366,0.290323,0.322581,0.322581,0.290323,3.423160,3.444693,3.405300,3.414672
2364,71,0.435651,Expert,PO3,0.222476,0.188065,0.039012,0.203466,0.079452,0.037665,...,0.037877,0.203373,0.290323,0.322581,0.290323,0.354839,3.407981,3.445670,3.427560,3.413501
2365,71,0.435651,Expert,POz,0.214923,0.176492,0.039372,0.211699,0.073116,0.031213,...,0.037066,0.210536,0.354839,0.290323,0.322581,0.387097,3.413788,3.449976,3.426663,3.416950
2366,71,0.435651,Expert,PO4,0.164153,0.175065,0.050551,0.213906,0.052657,0.030806,...,0.049701,0.205694,0.387097,0.387097,0.258065,0.322581,3.418935,3.450560,3.443043,3.403913


### 2.1.2 Baseline Data

In [29]:
#get features for code comprehension data
df_elec_baseline_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore', 'SkillLevel', 'Channel', 
            'Alpha_Mean', 'Beta_Mean', 'Gamma_Mean', 'Theta_Mean',
            'Alpha_StdDev', 'Beta_StdDev', 'Gamma_StdDev', 'Theta_StdDev',
            'Alpha_Skewness', 'Beta_Skewness', 'Gamma_Skewness', 'Theta_Skewness',
            'Alpha_Kurtosis', 'Beta_Kurtosis', 'Gamma_Kurtosis', 'Theta_Kurtosis',
            'Alpha_Variance', 'Beta_Variance', 'Gamma_Variance', 'Theta_Variance',
            'Alpha_Median', 'Beta_Median', 'Gamma_Median', 'Theta_Median',
            'Alpha_ZeroCrossRate', 'Beta_ZeroCrossRate', 'Gamma_ZeroCrossRate', 'Theta_ZeroCrossRate',
            'Alpha_Entropy', 'Beta_Entropy', 'Gamma_Entropy', 'Theta_Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_alpha = np.zeros(shape=(64, 0))
    merged_beta = np.zeros(shape=(64, 0))
    merged_gamma = np.zeros(shape=(64, 0))
    merged_theta = np.zeros(shape=(64, 0))

    for idx, row in df_participant.iterrows():
        alpha = row["BaselineAlpha"]
        beta = row["BaselineBeta"]
        gamma = row["BaselineGamma"]
        theta = row["BaselineTheta"]

        merged_alpha = np.concatenate((merged_alpha, alpha), axis=1)
        merged_beta = np.concatenate((merged_beta, beta), axis=1)
        merged_gamma = np.concatenate((merged_gamma, gamma), axis=1)
        merged_theta = np.concatenate((merged_theta, theta), axis=1)
    # Calculate statistical features for each channel
    for idx, channel in enumerate(channel_names):
        alpha_channeled = merged_alpha[idx, :]
        beta_channeled = merged_beta[idx, :]
        gamma_channeled = merged_gamma[idx, :]
        theta_channeled = merged_theta[idx, :]
        
        # Use the calculate_channel_statistics function on each frequency band
        alpha_stats = get_freqband_features(alpha_channeled)
        beta_stats = get_freqband_features(beta_channeled)
        gamma_stats = get_freqband_features(gamma_channeled)
        theta_stats = get_freqband_features(theta_channeled)

        
        
        # Create a DataFrame for the current channel
        channel_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Channel': channel,
            'Alpha_Mean': alpha_stats[0],
            'Beta_Mean': beta_stats[0],
            'Gamma_Mean': gamma_stats[0],
            'Theta_Mean': theta_stats[0],
            'Alpha_StdDev': alpha_stats[1],
            'Beta_StdDev': beta_stats[1],
            'Gamma_StdDev': gamma_stats[1],
            'Theta_StdDev': theta_stats[1],
            'Alpha_Skewness': alpha_stats[2],
            'Beta_Skewness': beta_stats[2],
            'Gamma_Skewness': gamma_stats[2],
            'Theta_Skewness': theta_stats[2],
            'Alpha_Kurtosis': alpha_stats[3],
            'Beta_Kurtosis': beta_stats[3],
            'Gamma_Kurtosis': gamma_stats[3],
            'Theta_Kurtosis': theta_stats[3],
            'Alpha_Variance': alpha_stats[4],
            'Beta_Variance': beta_stats[4],
            'Gamma_Variance': gamma_stats[4],
            'Theta_Variance': theta_stats[4],
            'Alpha_Median': alpha_stats[5],
            'Beta_Median': beta_stats[5],
            'Gamma_Median': gamma_stats[5],
            'Theta_Median': theta_stats[5],
            'Alpha_ZeroCrossRate': alpha_stats[6],
            'Beta_ZeroCrossRate': beta_stats[6],
            'Gamma_ZeroCrossRate': gamma_stats[6],
            'Theta_ZeroCrossRate': theta_stats[6],
            'Alpha_Entropy': alpha_stats[7],
            'Beta_Entropy': beta_stats[7],
            'Gamma_Entropy': gamma_stats[7],
            'Theta_Entropy': theta_stats[7]
        }, index=[0])

        df_elec_baseline_brain_waves_stats = pd.concat([df_elec_baseline_brain_waves_stats, channel_df], ignore_index=True)

df_elec_baseline_brain_waves_stats.to_csv(result_path+'/features_elec_abgtbw_bldata.csv')
df_elec_baseline_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\3577530283.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_elec_baseline_brain_waves_stats = pd.concat([df_elec_baseline_brain_waves_stats, channel_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Channel,Alpha_Mean,Beta_Mean,Gamma_Mean,Theta_Mean,Alpha_StdDev,Beta_StdDev,...,Gamma_Median,Theta_Median,Alpha_ZeroCrossRate,Beta_ZeroCrossRate,Gamma_ZeroCrossRate,Theta_ZeroCrossRate,Alpha_Entropy,Beta_Entropy,Gamma_Entropy,Theta_Entropy
0,1,0.331385,Intermediate,Fp1,0.395663,0.113486,0.063130,0.097166,0.169601,0.034206,...,0.057804,0.099865,0.354839,0.258065,0.290323,0.322581,3.373729,3.420319,3.412292,3.380473
1,1,0.331385,Intermediate,Fp2,0.369195,0.144116,0.063106,0.129108,0.173634,0.039341,...,0.060817,0.125693,0.322581,0.258065,0.258065,0.290323,3.363019,3.427183,3.409783,3.383050
2,1,0.331385,Intermediate,F7,0.292649,0.153357,0.077229,0.130586,0.114222,0.036661,...,0.076072,0.132598,0.322581,0.258065,0.322581,0.258065,3.386727,3.437070,3.449016,3.417734
3,1,0.331385,Intermediate,F3,0.325351,0.135161,0.055520,0.159799,0.129947,0.033203,...,0.057870,0.152070,0.322581,0.322581,0.354839,0.354839,3.388634,3.434490,3.420064,3.403502
4,1,0.331385,Intermediate,Fz,0.330490,0.129424,0.048896,0.177055,0.128879,0.030626,...,0.048574,0.174883,0.322581,0.290323,0.290323,0.290323,3.392261,3.435388,3.417281,3.407822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2363,71,0.435651,Expert,PO7,0.243428,0.294202,0.042262,0.130581,0.068930,0.063385,...,0.035524,0.137397,0.322581,0.290323,0.322581,0.354839,3.425648,3.441235,3.348820,3.405714
2364,71,0.435651,Expert,PO3,0.343685,0.210607,0.036058,0.132674,0.116240,0.057444,...,0.033280,0.125939,0.258065,0.290323,0.290323,0.387097,3.407349,3.430682,3.390230,3.413260
2365,71,0.435651,Expert,POz,0.321854,0.196170,0.039475,0.133307,0.123259,0.055767,...,0.034051,0.125352,0.322581,0.354839,0.322581,0.322581,3.390598,3.425174,3.372426,3.403717
2366,71,0.435651,Expert,PO4,0.278317,0.231861,0.047093,0.132904,0.098140,0.048063,...,0.042499,0.124243,0.322581,0.354839,0.387097,0.354839,3.402631,3.444007,3.397428,3.407373


## 2.2 Brain Waves --> 4 to 50 Hz range

### 2.2.1 Code Comprehension Data

In [30]:
#get features for code comprehension data
df_elec_4to50Hz_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore', 'SkillLevel', 'Channel', 
            'Mean', 'StdDev', 'Skewness', 'Kurtosis', 'Variance', 'Median', 'ZeroCrossRate', 'Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_full_spectrum = np.zeros(shape=(64, 0))
    
    for idx, row in df_participant.iterrows():
        full_spectrum = row["Range4to50Hz"]

        merged_full_spectrum = np.concatenate((merged_full_spectrum, full_spectrum), axis=1)

    for idx, channel in enumerate(channel_names):
        full_spectrum_channeled = merged_full_spectrum[idx, :]
        
        # Use the calculate_channel_statistics function on each frequency band
        full_spectrum_stats = get_freqband_features(full_spectrum_channeled)
        
        # Create a DataFrame for the current channel
        channel_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Channel': channel,
            'Mean': full_spectrum_stats[0],
            'StdDev': full_spectrum_stats[1],
            'Skewness': full_spectrum_stats[2],
            'Kurtosis': full_spectrum_stats[3],
            'Variance': full_spectrum_stats[4],
            'Median': full_spectrum_stats[5],
            'ZeroCrossRate': full_spectrum_stats[6],
            'Entropy': full_spectrum_stats[7],
        }, index=[0])

        df_elec_4to50Hz_brain_waves_stats = pd.concat([df_elec_4to50Hz_brain_waves_stats, channel_df], ignore_index=True)

df_elec_4to50Hz_brain_waves_stats.to_csv(result_path+'/features_elec_4to50hzbw_ccdata.csv')
df_elec_4to50Hz_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\995028288.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_elec_4to50Hz_brain_waves_stats = pd.concat([df_elec_4to50Hz_brain_waves_stats, channel_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Channel,Mean,StdDev,Skewness,Kurtosis,Variance,Median,ZeroCrossRate,Entropy
0,1,0.331385,Intermediate,Fp1,0.578004,0.060779,0.187927,-0.328752,0.003694,0.574458,0.290323,3.460217
1,1,0.331385,Intermediate,Fp2,0.630075,0.057554,-0.270284,-0.113730,0.003312,0.629625,0.322581,3.461512
2,1,0.331385,Intermediate,F7,0.603259,0.069906,0.057028,-0.523495,0.004887,0.596230,0.258065,3.458999
3,1,0.331385,Intermediate,F3,0.610693,0.090608,-0.509353,-0.256388,0.008210,0.609872,0.290323,3.454321
4,1,0.331385,Intermediate,Fz,0.625720,0.094338,-0.477489,-0.355320,0.008900,0.628981,0.258065,3.453964
...,...,...,...,...,...,...,...,...,...,...,...,...
2363,71,0.435651,Expert,PO7,0.685996,0.090573,0.128294,-0.056459,0.008204,0.688518,0.225806,3.456994
2364,71,0.435651,Expert,PO3,0.670809,0.082813,-0.051730,-0.780736,0.006858,0.669534,0.354839,3.458055
2365,71,0.435651,Expert,POz,0.661130,0.088330,0.496353,-0.132532,0.007802,0.646663,0.322581,3.456936
2366,71,0.435651,Expert,PO4,0.618289,0.093164,0.653408,0.204404,0.008680,0.603363,0.258065,3.454633


### 2.2.2 Baseline Data

In [31]:
#get features for code comprehension data
df_elec_bl_4to50Hz_brain_waves_stats = pd.DataFrame(
    columns=['Participant', 'SkillScore', 'SkillLevel', 'Channel', 
            'Mean', 'StdDev', 'Skewness', 'Kurtosis', 'Variance', 'Median', 'ZeroCrossRate', 'Entropy'])

for participant in tqdm(df_brain_waves["Participant"].unique(), total=len(df_brain_waves["Participant"].unique())):
    df_participant = df_brain_waves[df_brain_waves["Participant"] == participant]
    skill_score = df_participant["SkillScore"].unique()[0]
    skill_level = df_participant["SkillLevel"].unique()[0]

    merged_full_spectrum = np.zeros(shape=(64, 0))
    
    for idx, row in df_participant.iterrows():
        full_spectrum = row["BaselineRange4to50Hz"]

        merged_full_spectrum = np.concatenate((merged_full_spectrum, full_spectrum), axis=1)

    for idx, channel in enumerate(channel_names):
        full_spectrum_channeled = merged_full_spectrum[idx, :]
        
        # Use the calculate_channel_statistics function on each frequency band
        full_spectrum_stats = get_freqband_features(full_spectrum_channeled)
        
        # Create a DataFrame for the current channel
        channel_df = pd.DataFrame({
            'Participant': participant,
            'SkillScore': skill_score,
            'SkillLevel': skill_level,
            'Channel': channel,
            'Mean': full_spectrum_stats[0],
            'StdDev': full_spectrum_stats[1],
            'Skewness': full_spectrum_stats[2],
            'Kurtosis': full_spectrum_stats[3],
            'Variance': full_spectrum_stats[4],
            'Median': full_spectrum_stats[5],
            'ZeroCrossRate': full_spectrum_stats[6],
            'Entropy': full_spectrum_stats[7],
        }, index=[0])

        df_elec_bl_4to50Hz_brain_waves_stats = pd.concat([df_elec_bl_4to50Hz_brain_waves_stats, channel_df], ignore_index=True)

df_elec_bl_4to50Hz_brain_waves_stats.to_csv(result_path+'/features_elec_4to50hzbw_bldata.csv')
df_elec_bl_4to50Hz_brain_waves_stats

  0%|          | 0/37 [00:00<?, ?it/s]

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_19436\3052634333.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_elec_bl_4to50Hz_brain_waves_stats = pd.concat([df_elec_bl_4to50Hz_brain_waves_stats, channel_df], ignore_index=True)


,Participant,SkillScore,SkillLevel,Channel,Mean,StdDev,Skewness,Kurtosis,Variance,Median,ZeroCrossRate,Entropy
0,1,0.331385,Intermediate,Fp1,0.681020,0.108926,0.009201,-0.338011,0.011865,0.680778,0.354839,3.452806
1,1,0.331385,Intermediate,Fp2,0.718634,0.098006,0.531828,0.046404,0.009605,0.700738,0.322581,3.456580
2,1,0.331385,Intermediate,F7,0.664859,0.072894,-0.240494,-0.108777,0.005314,0.679650,0.387097,3.459637
3,1,0.331385,Intermediate,F3,0.688845,0.085296,-0.231119,-0.574746,0.007275,0.707573,0.290323,3.457946
4,1,0.331385,Intermediate,Fz,0.700076,0.087189,-0.392213,-0.166185,0.007602,0.713150,0.225806,3.457792
...,...,...,...,...,...,...,...,...,...,...,...,...
2363,71,0.435651,Expert,PO7,0.727753,0.080717,-0.088966,-0.996187,0.006515,0.730303,0.290323,3.459539
2364,71,0.435651,Expert,PO3,0.741144,0.077292,0.084621,-0.888981,0.005974,0.729622,0.322581,3.460293
2365,71,0.435651,Expert,POz,0.710518,0.094018,-0.485334,-0.883527,0.008839,0.730234,0.354839,3.456732
2366,71,0.435651,Expert,PO4,0.710536,0.091691,-0.453584,-0.671719,0.008407,0.725776,0.354839,3.457188


# Save the Result 

In [32]:
# List all files in the folder
files = os.listdir(result_path)

# Filter files starting with "features" and ending with ".csv"
feature_files = [file for file in files if file.startswith('features') and file.endswith('.csv')]

# Check if any matching files were found
if not feature_files:
    print("No matching CSV files found.")
else:
    # Initialize an empty list to store data
    df_processed = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','FilePath'])

    # Loop through each matching file
    for feature_file in feature_files:
        # Construct the full path to the CSV file
        csv_path = os.path.join(result_path, feature_file)

        # Extract information from the file name
        file_info = feature_file.split('_')
        focus = file_info[1]
        frequency_band = file_info[2].replace('bw', '') if len(file_info) > 2 else None
        datatype = file_info[3][:2] if len(file_info) > 3 else None

        #Clear Naming 
        if datatype == 'bl':
            datatype = 'Baseline'
        elif datatype == 'cc':
            datatype = 'CodeComprehension'

        if focus == 'alg':
            focus = 'Algorithm'
        elif focus == 'elec':
            focus = 'ElectrodePosition'

        if frequency_band == 'abgt':
            frequency_band = 'AlphaBetaThetaGamma'
        # Append data to the list
        df_processed.loc[len(df_processed)]= [focus, datatype, frequency_band, csv_path  ]

df_processed.to_csv(result_path + '/processed_data.csv')
df_processed

,Focus,Data,FrequencyBand,FilePath
0,Algorithm,Baseline,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
1,Algorithm,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
2,Algorithm,Baseline,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
3,Algorithm,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
4,ElectrodePosition,Baseline,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
5,ElectrodePosition,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
6,ElectrodePosition,Baseline,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
7,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
